In [24]:
import pandas as pd
import numpy as np
import re

raw_path = "/Users/ronnygottheway/Desktop/美赛/data/raw/2026_MCM_Problem_C_Data.csv"
feat_path = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/data_with_features_engineered-1.xlsx"

df = pd.read_csv(raw_path)
print(df.shape)
print(df.columns[:20])

(421, 53)
Index(['celebrity_name', 'ballroom_partner', 'celebrity_industry',
       'celebrity_homestate', 'celebrity_homecountry/region',
       'celebrity_age_during_season', 'season', 'results', 'placement',
       'week1_judge1_score', 'week1_judge2_score', 'week1_judge3_score',
       'week1_judge4_score', 'week2_judge1_score', 'week2_judge2_score',
       'week2_judge3_score', 'week2_judge4_score', 'week3_judge1_score',
       'week3_judge2_score', 'week3_judge3_score'],
      dtype='object')


In [18]:
# 1) 找到所有评委分列 weekX_judgeY_score
score_cols = []
pattern = re.compile(r"^week(\d+)_judge(\d+)_score$")

for c in df.columns:
    if pattern.match(c):
        score_cols.append(c)

print("n_score_cols =", len(score_cols))
print("example:", score_cols[:8])

# 2) 强制转成数值（避免字符串导致 replace 不生效）
df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")

# 3) 只在评委分列上：0 -> NaN
df[score_cols] = df[score_cols].replace(0, np.nan)

# 4) 验证：这些列里不应再有 0
zeros_after = (df[score_cols] == 0).sum().sum()
print("zeros_after =", zeros_after)



n_score_cols = 44
example: ['week1_judge1_score', 'week1_judge2_score', 'week1_judge3_score', 'week1_judge4_score', 'week2_judge1_score', 'week2_judge2_score', 'week2_judge3_score', 'week2_judge4_score']
zeros_after = 0


In [19]:
## 你需要确保这些 ID 列在原始数据里存在
# 如果列名不同（比如 celebrity 而不是 celebrity_name），请按实际列名改这里
id_cols = ["season", "celebrity_name", "ballroom_partner"]

missing = [c for c in id_cols if c not in df.columns]
if missing:
    raise ValueError(f"raw data 缺少这些列：{missing}；请检查原始列名并调整 id_cols。")

df_scores = df[id_cols + score_cols].melt(
    id_vars=id_cols,
    value_vars=score_cols,
    var_name="score_col",
    value_name="score"
)

# 从 score_col 提取 week, judge
df_scores[["week", "judge"]] = df_scores["score_col"].str.extract(r"^week(\d+)_judge(\d+)_score$")
df_scores["week"] = df_scores["week"].astype(int)
df_scores["judge"] = df_scores["judge"].astype(int)

print(df_scores.shape)
print(df_scores.head())



(18524, 7)
   season     celebrity_name     ballroom_partner           score_col  score  \
0       1      John O'Hurley  Charlotte Jorgensen  week1_judge1_score    7.0   
1       1       Kelly Monaco            Alec Mazo  week1_judge1_score    5.0   
2       1  Evander Holyfield      Edyta Sliwinska  week1_judge1_score    5.0   
3       1      Rachel Hunter     Jonathan Roberts  week1_judge1_score    7.0   
4       1      Joey McIntyre      Ashly DelGrosso  week1_judge1_score    7.0   

   week  judge  
0     1      1  
1     1      1  
2     1      1  
3     1      1  
4     1      1  


In [20]:
# ddof=0 是总体方差；你也可以改成 ddof=1（样本方差），但全程一致即可
agg = df_scores.groupby(id_cols + ["week"], as_index=False).agg(
    J_total=("score", "sum"),
    n_judges=("score", lambda x: x.notna().sum()),
    J_mean_per_judge=("score", "mean"),
    variance=("score", lambda x: np.nanvar(x.values, ddof=0))
)

agg["is_active"] = (agg["n_judges"] > 0).astype(int)

# 争议阈值（你之前用 0.68 就先沿用）
THRESHOLD = 0.68
agg["variance_contr"] = ((agg["variance"] > THRESHOLD) & (agg["n_judges"] >= 2)).astype(int)

# group_id：Ranker 按周分组用
agg["group_id"] = "S" + agg["season"].astype(str) + "_W" + agg["week"].astype(str)

print(agg.shape)
print(agg.head())


(4631, 11)
   season     celebrity_name ballroom_partner  week  J_total  n_judges  \
0       1  Evander Holyfield  Edyta Sliwinska     1     18.0         3   
1       1  Evander Holyfield  Edyta Sliwinska     2     14.0         3   
2       1  Evander Holyfield  Edyta Sliwinska     3     13.0         3   
3       1  Evander Holyfield  Edyta Sliwinska     4      0.0         0   
4       1  Evander Holyfield  Edyta Sliwinska     5      0.0         0   

   J_mean_per_judge  variance  is_active  variance_contr group_id  
0          6.000000  0.666667          1               0    S1_W1  
1          4.666667  0.222222          1               0    S1_W2  
2          4.333333  0.222222          1               0    S1_W3  
3               NaN       NaN          0               0    S1_W4  
4               NaN       NaN          0               0    S1_W5  


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_2073/1844383407.py:6: RuntimeWarning: Degrees of freedom <= 0 for slice.
  variance=("score", lambda x: np.nanvar(x.values, ddof=0))


In [21]:
# 排序保证滚动计算正确
agg = agg.sort_values(["season", "celebrity_name", "week"]).reset_index(drop=True)

# 不含本周：先 shift(1)，再对过去值做 expanding mean
agg["rolling_score"] = (
    agg.groupby(["season", "celebrity_name"])["J_mean_per_judge"]
       .apply(lambda s: s.shift(1).expanding().mean())
       .reset_index(level=[0,1], drop=True)
)

# 可选：你也可以同时保留一个“含本周”的版本（如果你们设定投票发生在评分后）
agg["rolling_score_including_t"] = (
    agg.groupby(["season", "celebrity_name"])["J_mean_per_judge"]
       .apply(lambda s: s.expanding().mean())
       .reset_index(level=[0,1], drop=True)
)

print(agg[["season","celebrity_name","week","J_mean_per_judge","rolling_score","rolling_score_including_t"]].head(12))


    season     celebrity_name  week  J_mean_per_judge  rolling_score  \
0        1  Evander Holyfield     1          6.000000            NaN   
1        1  Evander Holyfield     2          4.666667       6.000000   
2        1  Evander Holyfield     3          4.333333       5.333333   
3        1  Evander Holyfield     4               NaN       5.000000   
4        1  Evander Holyfield     5               NaN       5.000000   
5        1  Evander Holyfield     6               NaN       5.000000   
6        1  Evander Holyfield     7               NaN       5.000000   
7        1  Evander Holyfield     8               NaN       5.000000   
8        1  Evander Holyfield     9               NaN       5.000000   
9        1  Evander Holyfield    10               NaN       5.000000   
10       1  Evander Holyfield    11               NaN       5.000000   
11       1      Joey McIntyre     1          6.666667            NaN   

    rolling_score_including_t  
0                    6.000000  

In [27]:
feat = pd.read_excel(feat_path)
print(feat.shape)
print(feat.columns[:30])

# 自动选取你要的列
need_base = ["season", "celebrity_name", "ballroom_partner", "normalized_weight", "trend_late_minus_early"]

homeland_cols = [c for c in feat.columns if c.startswith("celebrity_homeland_")]
industry_cols = [c for c in feat.columns if c.startswith("celebrity_industry_")]

need_cols = [c for c in need_base if c in feat.columns] + homeland_cols + industry_cols

missing_need = [c for c in need_base if c not in feat.columns]
if missing_need:
    raise ValueError(f"features 表缺少这些关键列：{missing_need}；请检查 features 文件列名。")

feat_use = feat[need_cols].copy()

# merge：建议 key 用 (season, celebrity_name, ballroom_partner)
merged = agg.merge(
    feat_use,
    how="left",
    on=["season", "celebrity_name", "ballroom_partner"]
)

print(merged.shape)
print("merge 后 normalized_weight 缺失比例：", merged["normalized_weight"].isna().mean())


(421, 113)
Index(['celebrity_name', 'ballroom_partner', 'celebrity_age_during_season',
       'mean_score', 'std_score', 'trend_late_minus_early',
       'normalized_weight', 'season', 'celebrity_industry_Astronaut',
       'celebrity_industry_Athlete', 'celebrity_industry_Beauty Pagent',
       'celebrity_industry_Comedian', 'celebrity_industry_Con artist',
       'celebrity_industry_Conservationist', 'celebrity_industry_Entrepreneur',
       'celebrity_industry_Fashion Designer',
       'celebrity_industry_Fitness Instructor',
       'celebrity_industry_Journalist', 'celebrity_industry_Magician',
       'celebrity_industry_Military', 'celebrity_industry_Model',
       'celebrity_industry_Motivational Speaker',
       'celebrity_industry_Musician', 'celebrity_industry_News Anchor',
       'celebrity_industry_Politician', 'celebrity_industry_Producer',
       'celebrity_industry_Racing Driver',
       'celebrity_industry_Radio Personality',
       'celebrity_industry_Singer/Rapper',
  

In [28]:
keep_cols = (
    ["season", "week", "group_id", "celebrity_name", "ballroom_partner",
     "is_active",
     "J_total", "n_judges", "J_mean_per_judge",
     "variance", "variance_contr",
     "rolling_score",
     "trend_late_minus_early",
     "normalized_weight"]
    + homeland_cols
    + industry_cols
)

final = merged[keep_cols].copy()

# 只保留在场行（不在场的行对周内比较没意义；也可保留但训练时过滤）
final_active = final[final["is_active"] == 1].reset_index(drop=True)

out_path = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_long_for_ranker.csv"
final_active.to_csv(out_path, index=False)

print(final_active.shape)
print("saved to:", out_path)


(2777, 108)
saved to: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_long_for_ranker.csv


In [29]:
import pandas as pd
import numpy as np

path = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_long_for_ranker.csv"
df = pd.read_csv(path)

# 0) 基础检查
need = ["season","week","group_id","celebrity_name","J_total"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"缺少关键列: {missing}")

df = df.sort_values(["season","celebrity_name","week"]).reset_index(drop=True)

# 1) 标记“下周是否仍在场”：用自连接（向量化，不用 apply）
df_next = df[["season","celebrity_name","week"]].copy()
df_next["week"] = df_next["week"] - 1   # 让“下周的出现”对齐到“本周”
df_next["has_next_week"] = 1

df = df.merge(df_next, on=["season","celebrity_name","week"], how="left")
df["active_next_week"] = df["has_next_week"].fillna(0).astype(int)
df.drop(columns=["has_next_week"], inplace=True)

# 2) 生成 eliminated_this_week（排除每个赛季的最后一周：最后一周没有“下周”可判别）
max_week = df.groupby("season")["week"].transform("max")
df["eliminated_this_week"] = ((df["active_next_week"] == 0) & (df["week"] < max_week)).astype(int)

# 3) 计算每周淘汰人数 + week_type
week_elims = df.groupby(["season","week"], as_index=False)["eliminated_this_week"].sum()
week_elims.rename(columns={"eliminated_this_week":"n_eliminated_week"}, inplace=True)

week_elims["week_type"] = np.select(
    [week_elims["n_eliminated_week"] == 0,
     week_elims["n_eliminated_week"] == 1,
     week_elims["n_eliminated_week"] == 2],
    ["no_elim", "single", "double"],
    default="multi"
)

df = df.merge(week_elims, on=["season","week"], how="left")

# 4) 赛制段 rule_segment（按你们规则：1-2 rank；3-27 percent；28+ save）
df["rule_segment"] = np.select(
    [df["season"].isin([1,2]),
     (df["season"] >= 3) & (df["season"] <= 27),
     df["season"] >= 28],
    ["rank","percent","save"],
    default="unknown"
)

# 5) percent 赛制需要：sum_J_week + judge_percent
df["sum_J_week"] = df.groupby("group_id")["J_total"].transform("sum")
df["judge_percent"] = df["J_total"] / df["sum_J_week"]

# 6) 保存
out = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_long_for_ranker_labeled.csv"
df.to_csv(out, index=False)
print("saved:", out)
print(df[["season","week","week_type","n_eliminated_week","rule_segment"]].head(10))


saved: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_long_for_ranker_labeled.csv
   season  week week_type  n_eliminated_week rule_segment
0       1     1   no_elim                  0         rank
1       1     2    single                  1         rank
2       1     3    single                  1         rank
3       1     1   no_elim                  0         rank
4       1     2    single                  1         rank
5       1     3    single                  1         rank
6       1     4    single                  1         rank
7       1     5    single                  1         rank
8       1     1   no_elim                  0         rank
9       1     2    single                  1         rank


In [2]:
# 看看哪些特征列是 object
obj_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
print("object cols:", obj_cols)

# 看看这些列的示例值
for c in obj_cols[:20]:
    print(c, "->", X_tr[c].head(3).tolist())


object cols: ['celebrity_homeland_Alaska', 'celebrity_homeland_Argentina', 'celebrity_homeland_Arizona', 'celebrity_homeland_Arkansas', 'celebrity_homeland_Australia', 'celebrity_homeland_Brazil', 'celebrity_homeland_California', 'celebrity_homeland_Canada', 'celebrity_homeland_Chile', 'celebrity_homeland_Colorado', 'celebrity_homeland_Connecticut', 'celebrity_homeland_Croatia', 'celebrity_homeland_Cuba', 'celebrity_homeland_Czechoslovakia', 'celebrity_homeland_Delaware', 'celebrity_homeland_England', 'celebrity_homeland_Florida', 'celebrity_homeland_France', 'celebrity_homeland_Georgia', 'celebrity_homeland_Germany', 'celebrity_homeland_Hawaii', 'celebrity_homeland_Illinois', 'celebrity_homeland_Indiana', 'celebrity_homeland_Iowa', 'celebrity_homeland_Ireland', 'celebrity_homeland_Italy', 'celebrity_homeland_Kansas', 'celebrity_homeland_Kentucky', 'celebrity_homeland_Louisiana', 'celebrity_homeland_Maine', 'celebrity_homeland_Maryland', 'celebrity_homeland_Massachusetts', 'celebrity_h

In [3]:
# 1) 把 bool 列转成 int（True/False -> 1/0）
bool_cols = X_tr.select_dtypes(include=["bool"]).columns.tolist()
X_tr[bool_cols] = X_tr[bool_cols].astype(int)
X_va[bool_cols] = X_va[bool_cols].astype(int)

# 2) 把 object 列里如果也是 True/False/0/1 字符串，强制数值化
X_tr = X_tr.apply(pd.to_numeric, errors="coerce")
X_va = X_va.apply(pd.to_numeric, errors="coerce")

# 3) 若仍残留 object（极少），直接删除
bad_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
if bad_cols:
    print("Dropping non-numeric columns:", bad_cols)
    X_tr = X_tr.drop(columns=bad_cols)
    X_va = X_va.drop(columns=bad_cols)


In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb

path = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_long_for_ranker_labeled.csv"
df = pd.read_csv(path)

# 只训练 single/double
train_df = df[df["week_type"].isin(["single","double"])].copy()
train_df["relevance"] = np.where(train_df["eliminated_this_week"] == 1, 0, 1)
train_df = train_df.sort_values(["group_id", "celebrity_name"]).reset_index(drop=True)

# season 留出验证（最后两季）
seasons = sorted(train_df["season"].unique())
valid_seasons = seasons[-2:]
tr = train_df[~train_df["season"].isin(valid_seasons)].copy()
va = train_df[ train_df["season"].isin(valid_seasons)].copy()

tr = tr.sort_values(["group_id","celebrity_name"]).reset_index(drop=True)
va = va.sort_values(["group_id","celebrity_name"]).reset_index(drop=True)

tr_group = tr.groupby("group_id").size().to_numpy()
va_group = va.groupby("group_id").size().to_numpy()

drop_cols = [
    "season","week","group_id","celebrity_name","ballroom_partner",
    "is_active","active_next_week",
    "eliminated_this_week","n_eliminated_week","week_type","rule_segment",
    "sum_J_week","judge_percent",
    "relevance"
]
feature_cols = [c for c in tr.columns if c not in drop_cols]

X_tr = tr[feature_cols].copy()
X_va = va[feature_cols].copy()
y_tr = tr["relevance"]
y_va = va["relevance"]

# ——关键：统一 one-hot 类型（True/False/字符串 → 0/1）——
for X in (X_tr, X_va):
    X.replace({True:1, False:0, "True":1, "False":0}, inplace=True)

X_tr = X_tr.apply(pd.to_numeric, errors="coerce")
X_va = X_va.apply(pd.to_numeric, errors="coerce")

bad_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
if bad_cols:
    print("Dropping non-numeric cols:", bad_cols)
    X_tr = X_tr.drop(columns=bad_cols)
    X_va = X_va.drop(columns=bad_cols)

print("object cols left:", X_tr.select_dtypes(include=["object"]).columns.tolist())
print("X_tr shape:", X_tr.shape, "X_va shape:", X_va.shape)


object cols left: []
X_tr shape: (2101, 102) X_va shape: (131, 102)


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_3968/2249778016.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.replace({True:1, False:0, "True":1, "False":0}, inplace=True)


In [10]:
ranker = xgb.XGBRanker(
    objective="rank:pairwise",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

ranker.fit(
    X_tr, y_tr,
    group=tr_group,
    eval_set=[(X_va, y_va)],
    eval_group=[va_group],
    verbose=50
)

model_path = "/Users/ronnygottheway/Desktop/美赛/models/xgbranker.json"

import os
os.makedirs(os.path.dirname(model_path), exist_ok=True)

ranker.save_model(model_path)
print("model saved:", model_path)


[0]	validation_0-ndcg@32:0.99148
[50]	validation_0-ndcg@32:0.98412
[100]	validation_0-ndcg@32:0.98343
[150]	validation_0-ndcg@32:0.98356
[200]	validation_0-ndcg@32:0.98391
[250]	validation_0-ndcg@32:0.98391
[300]	validation_0-ndcg@32:0.98391
[350]	validation_0-ndcg@32:0.98391
[400]	validation_0-ndcg@32:0.98391
[450]	validation_0-ndcg@32:0.98362
[500]	validation_0-ndcg@32:0.98362
[550]	validation_0-ndcg@32:0.98412
[600]	validation_0-ndcg@32:0.98412
[650]	validation_0-ndcg@32:0.98412
[700]	validation_0-ndcg@32:0.98412
[750]	validation_0-ndcg@32:0.98412
[800]	validation_0-ndcg@32:0.98412
[850]	validation_0-ndcg@32:0.98412
[900]	validation_0-ndcg@32:0.98412
[950]	validation_0-ndcg@32:0.98412
[1000]	validation_0-ndcg@32:0.98412
[1050]	validation_0-ndcg@32:0.98412
[1100]	validation_0-ndcg@32:0.98412
[1150]	validation_0-ndcg@32:0.98412
[1200]	validation_0-ndcg@32:0.98412
[1250]	validation_0-ndcg@32:0.98412
[1300]	validation_0-ndcg@32:0.98412
[1350]	validation_0-ndcg@32:0.98412
[1400]	validati

In [11]:
import pandas as pd
import numpy as np

import xgboost as xgb

model_path = "/Users/ronnygottheway/Desktop/美赛/models/xgbranker.json"

ranker = xgb.XGBRanker()
ranker.load_model(model_path)
print("model loaded:", model_path)

# 1) 取训练时一致的特征列（非常关键）
feature_cols_used = list(X_tr.columns)

# 2) 构造全量特征矩阵
X_all = df[feature_cols_used].copy()

# 3) 同样的类型修复（保持与训练一致）
X_all = X_all.replace({True:1, False:0, "True":1, "False":0}).infer_objects(copy=False)
X_all = X_all.apply(pd.to_numeric, errors="coerce")

# 4) 预测打分 u_hat
df["u_hat"] = ranker.predict(X_all)

# 5) 保存
out_u = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/with_u_hat.csv"
df.to_csv(out_u, index=False)
print("saved:", out_u)

# 快速检查
print(df[["season","week","group_id","celebrity_name","u_hat"]].head())


model loaded: /Users/ronnygottheway/Desktop/美赛/models/xgbranker.json
saved: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/with_u_hat.csv
   season  week group_id     celebrity_name     u_hat
0       1     1    S1_W1  Evander Holyfield  1.613349
1       1     2    S1_W2  Evander Holyfield -3.194918
2       1     3    S1_W3  Evander Holyfield -5.916859
3       1     1    S1_W1      Joey McIntyre  4.455723
4       1     2    S1_W2      Joey McIntyre -0.034277


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_3968/154017707.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_all = X_all.replace({True:1, False:0, "True":1, "False":0}).infer_objects(copy=False)


In [12]:
import numpy as np

def softmax_vec(u, tau=1.0):
    u = np.asarray(u, dtype=float)
    u = u - np.nanmax(u)          # 数值稳定
    ex = np.exp(tau * u)
    return ex / np.nansum(ex)

TAU = 1.0

df = df.sort_values(["group_id", "celebrity_name"]).reset_index(drop=True)
df["q_hat"] = df.groupby("group_id")["u_hat"].transform(lambda s: softmax_vec(s.values, tau=TAU))

# 检查每周份额和为 1
print(df.groupby("group_id")["q_hat"].sum().describe())


count    3.350000e+02
mean     1.000000e+00
std      1.149419e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: q_hat, dtype: float64


In [13]:
import pandas as pd
import numpy as np

# percent：S = judge_percent + q_hat
df["S_score"] = df["judge_percent"] + df["q_hat"]

# rank：R = rank(J_total) + rank(q_hat)（1最好，越大越差）
def rank_desc_avg(s):
    return s.rank(ascending=False, method="average")

df["rank_J"] = df.groupby("group_id")["J_total"].transform(rank_desc_avg)
df["rank_F"] = df.groupby("group_id")["q_hat"].transform(rank_desc_avg)
df["R_score"] = df["rank_J"] + df["rank_F"]

# 预测淘汰：percent 段 S 最小；rank 段 R 最大
df["pred_elim_percent"] = (df.groupby("group_id")["S_score"].transform("min") == df["S_score"]).astype(int)
df["pred_elim_rank"]    = (df.groupby("group_id")["R_score"].transform("max") == df["R_score"]).astype(int)

# save 段 bottom2：S 最小两位
df["pred_bottom2_save"] = 0
save_mask = df["rule_segment"].eq("save")

def mark_bottom2(g):
    idx = g["S_score"].nsmallest(2).index
    g.loc[idx, "pred_bottom2_save"] = 1
    return g

df.loc[save_mask, :] = df.loc[save_mask, :].groupby("group_id", group_keys=False).apply(mark_bottom2)


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_3968/2798609248.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.loc[save_mask, :] = df.loc[save_mask, :].groupby("group_id", group_keys=False).apply(mark_bottom2)


In [14]:
eval_df = df[df["week_type"].isin(["single","double"])].copy()

# percent top-1 命中
mask_percent = eval_df["rule_segment"].eq("percent")
hit_percent = (
    (eval_df.loc[mask_percent, "pred_elim_percent"] & eval_df.loc[mask_percent, "eliminated_this_week"])
    .groupby(eval_df.loc[mask_percent, "group_id"]).any().mean()
)

# rank top-1 命中
mask_rank = eval_df["rule_segment"].eq("rank")
hit_rank = (
    (eval_df.loc[mask_rank, "pred_elim_rank"] & eval_df.loc[mask_rank, "eliminated_this_week"])
    .groupby(eval_df.loc[mask_rank, "group_id"]).any().mean()
)

# save bottom2 命中
save_eval = eval_df[eval_df["rule_segment"].eq("save")].copy()
hit_bottom2 = (
    (save_eval["pred_bottom2_save"] & save_eval["eliminated_this_week"])
    .groupby(save_eval["group_id"]).any().mean()
)

print("percent top-1 accuracy:", hit_percent)
print("rank top-1 accuracy:", hit_rank)
print("save bottom2 hit rate:", hit_bottom2)


percent top-1 accuracy: 0.5833333333333334
rank top-1 accuracy: 0.7272727272727273
save bottom2 hit rate: 0.7678571428571429


In [16]:
import numpy as np
import pandas as pd

taus = [0.25, 0.5, 1, 2, 5, 10]

# 选 percent 段存在的赛季：3-27
percent_seasons = sorted(df.loc[df["rule_segment"].eq("percent"), "season"].unique())
valid_percent = percent_seasons[-2:]   # 你也可以改成 [-3:] 更稳

val_p = df[df["season"].isin(valid_percent)].copy()

def softmax_group(series, tau):
    u = series.to_numpy(dtype=float)
    u = u - np.nanmax(u)
    ex = np.exp(tau * u)
    return ex / np.nansum(ex)

rows = []
for tau in taus:
    tmp = val_p.copy()
    tmp["q"] = tmp.groupby("group_id")["u_hat"].transform(lambda s: softmax_group(s, tau))
    tmp["S"] = tmp["judge_percent"] + tmp["q"]

    ev = tmp[tmp["week_type"].isin(["single","double"])].copy()
    # percent top-1
    pred = (ev.groupby("group_id")["S"].transform("min") == ev["S"]).astype(int)
    acc = (pred & ev["eliminated_this_week"]).groupby(ev["group_id"]).any().mean()

    rows.append((tau, acc))

res_p = pd.DataFrame(rows, columns=["tau","percent_top1"])
print("valid_percent seasons:", valid_percent)
print(res_p)


valid_percent seasons: [np.int64(26), np.int64(27)]
     tau  percent_top1
0   0.25      0.777778
1   0.50      0.666667
2   1.00      0.222222
3   2.00      0.111111
4   5.00      0.111111
5  10.00      0.111111


In [17]:
taus = [0.25, 0.5, 1, 2, 5, 10]
val_r = df[df["rule_segment"].eq("rank")].copy()

rows = []
for tau in taus:
    tmp = val_r.copy()
    tmp["q"] = tmp.groupby("group_id")["u_hat"].transform(lambda s: softmax_group(s, tau))

    tmp["rank_J"] = tmp.groupby("group_id")["J_total"].transform(lambda s: s.rank(ascending=False, method="average"))
    tmp["rank_F"] = tmp.groupby("group_id")["q"].transform(lambda s: s.rank(ascending=False, method="average"))
    tmp["R"] = tmp["rank_J"] + tmp["rank_F"]

    ev = tmp[tmp["week_type"].isin(["single","double"])].copy()
    pred = (ev.groupby("group_id")["R"].transform("max") == ev["R"]).astype(int)
    acc = (pred & ev["eliminated_this_week"]).groupby(ev["group_id"]).any().mean()

    rows.append((tau, acc))

res_r = pd.DataFrame(rows, columns=["tau","rank_top1"])
print(res_r)


     tau  rank_top1
0   0.25   0.727273
1   0.50   0.727273
2   1.00   0.727273
3   2.00   0.727273
4   5.00   0.727273
5  10.00   0.727273


In [18]:
import numpy as np
import pandas as pd

tau_percent = 0.25
tau_rank = 1.0
tau_save = 1.0

def softmax_vec(u, tau=1.0):
    u = np.asarray(u, dtype=float)
    u = u - np.nanmax(u)
    ex = np.exp(tau * u)
    return ex / np.nansum(ex)

df_final = df.copy()

# 分段温度：按 group_id 内 rule_segment 选择 tau，再算 q_hat
def apply_tau(g):
    seg = g["rule_segment"].iloc[0]
    if seg == "percent":
        tau = tau_percent
    elif seg == "rank":
        tau = tau_rank
    elif seg == "save":
        tau = tau_save
    else:
        tau = 1.0
    g["q_hat"] = softmax_vec(g["u_hat"].values, tau=tau)
    g["tau_used"] = tau
    return g

df_final = df_final.groupby("group_id", group_keys=False).apply(apply_tau)

# percent：S = judge_percent + q_hat
df_final["S_score"] = df_final["judge_percent"] + df_final["q_hat"]

# rank：R = rank(J_total) + rank(q_hat)
df_final["rank_J"] = df_final.groupby("group_id")["J_total"].transform(lambda s: s.rank(ascending=False, method="average"))
df_final["rank_F"] = df_final.groupby("group_id")["q_hat"].transform(lambda s: s.rank(ascending=False, method="average"))
df_final["R_score"] = df_final["rank_J"] + df_final["rank_F"]

# 预测淘汰 / bottom2
df_final["pred_elim_percent"] = (df_final.groupby("group_id")["S_score"].transform("min") == df_final["S_score"]).astype(int)
df_final["pred_elim_rank"]    = (df_final.groupby("group_id")["R_score"].transform("max") == df_final["R_score"]).astype(int)

df_final["pred_bottom2_save"] = 0
save_mask = df_final["rule_segment"].eq("save")

def mark_bottom2(g):
    idx = g["S_score"].nsmallest(2).index
    g.loc[idx, "pred_bottom2_save"] = 1
    return g

df_final.loc[save_mask, :] = df_final.loc[save_mask, :].groupby("group_id", group_keys=False).apply(mark_bottom2)

# 导出列（你们报告/提交表最常用）
out_cols = [
    "season","week","group_id","rule_segment","week_type",
    "celebrity_name","ballroom_partner",
    "u_hat","q_hat","tau_used",
    "J_total","judge_percent",
    "S_score","R_score",
    "pred_elim_percent","pred_elim_rank","pred_bottom2_save",
    "eliminated_this_week"
]

out_path = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_outputs_with_qhat.csv"
df_final[out_cols].to_csv(out_path, index=False)
print("saved:", out_path)


saved: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_outputs_with_qhat.csv


/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_3968/3068471990.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final = df_final.groupby("group_id", group_keys=False).apply(apply_tau)
/var/folders/6d/r8v3wkt15595tj6hq0kwh7rc0000gn/T/ipykernel_3968/3068471990.py:53: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final.loc[save_mask, :] = df_final.loc[save_mask, :].groupby("group_id", group_keys=Fa

In [19]:
eval_df = df_final[df_final["week_type"].isin(["single","double"])].copy()

# percent top-1
mask_percent = eval_df["rule_segment"].eq("percent")
hit_percent = (
    (eval_df.loc[mask_percent, "pred_elim_percent"] & eval_df.loc[mask_percent, "eliminated_this_week"])
    .groupby(eval_df.loc[mask_percent, "group_id"]).any().mean()
)

# rank top-1
mask_rank = eval_df["rule_segment"].eq("rank")
hit_rank = (
    (eval_df.loc[mask_rank, "pred_elim_rank"] & eval_df.loc[mask_rank, "eliminated_this_week"])
    .groupby(eval_df.loc[mask_rank, "group_id"]).any().mean()
)

# save bottom2
save_eval = eval_df[eval_df["rule_segment"].eq("save")].copy()
hit_bottom2 = (
    (save_eval["pred_bottom2_save"] & save_eval["eliminated_this_week"])
    .groupby(save_eval["group_id"]).any().mean()
)

print("FINAL percent top-1 accuracy:", hit_percent)
print("FINAL rank top-1 accuracy:", hit_rank)
print("FINAL save bottom2 hit rate:", hit_bottom2)


FINAL percent top-1 accuracy: 0.828125
FINAL rank top-1 accuracy: 0.7272727272727273
FINAL save bottom2 hit rate: 0.7678571428571429


In [20]:
import numpy as np

df_final["w_hat"] = np.exp(df_final["tau_used"] * df_final["u_hat"])


In [21]:
df_final["w_rel_mean1"] = df_final["w_hat"] / df_final.groupby("group_id")["w_hat"].transform("mean")


In [22]:
df_final["fan_percent"] = df_final["w_hat"] / df_final.groupby("group_id")["w_hat"].transform("sum")


In [23]:
df_final["fan_rank"] = df_final.groupby("group_id")["w_hat"].transform(
    lambda s: s.rank(ascending=False, method="average")
)


In [24]:
out_path = "/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_outputs_with_relative_votes.csv"
df_final.to_csv(out_path, index=False)
print("saved:", out_path)


saved: /Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_outputs_with_relative_votes.csv
